# Tutorial 6: Reasoning with Automated Planning

> **User Problem**
> A resident living in our smart home wants a new 'away' mode. Activating this mode must turn off all active lights and lock the main entrance door. Instead of programming the exact sequence of operators, it is easier for them to just describe the goal. 

In **Tutorial 5: World Model**, we introduced `WorldGraph` and `WorldSnapshot` to represent our smart home as a relational knowledge graph of rooms, topology, and device states. That enabled graph-grounded reasoning, but executing tasks still required manually scripting exact step-by-step operator sequences.

In this tutorial, you will extend the Smart Home Assistant to use **Automated Planning** as a reasoning mechanism. Rather than hardcoding procedural execution paths, you will define the target **goal state** (e.g., *all lights are off and the main door is locked*) and let `cognition` automatically discover, order, and execute the required action sequence.

## Step 1: Defining Causal World Model Transitions

In Tutorial 5, we built an internal model of the world using a knowledge graph (`WorldGraph`) and created an `Operator` to keep it updated. A `WorldSnapshot` provides safe, immutable access to world state information, enabling reasoning processes to deliberatively process and simulate possibilities without impacting the real environment. While the `WorldGraph` and `WorldSnapshot` are *descriptive*, i.e, they capture the current reality, `cognition` also enables developers to build *predictive* world models by specifying how the environment evolves when acted upon. 

We define these causal state transitions using `WorldEvolver` constructs, each implementing:
- **when** (*pre-conditions*) : Encodes common-sense logic determining when an action can be initiated given the current state. It inspects the `WorldSnapshot` knowledge graph and yields instances only when prerequisites and constraints are met.   
- **then** (*effects*): Captures what happens when that action is executed. It transforms the `WorldSnapshot` using immutable entity updates (`state.copy(add=[...], remove=[...])`) and returns the resulting simulated state along with an action execution cost (default `1`).   

We also connect the `WorldEvolver` with the relevant `Action` payload during __init__.  

Explore the `WorldEvolver`s below written for all possible actions in our Smart Home World. 

In [1]:
from typing import Iterable, cast
from cognition import SearchPlannerDynamicOption as WorldEvolver, WorldSnapshot
from smart_home_assistant_v6 import Device, DeviceName, DeviceState, DeviceType, Action, ControlSignal

class EvolverTurnLightOn(WorldEvolver[WorldSnapshot, Action]):
    def __init__(self, light: Device) -> None:
        super().__init__(Action(device=cast(DeviceName, light.name), signal=ControlSignal.TURN_ON))
        self._light: Device = light

    @classmethod
    def when(cls, state: WorldSnapshot) -> Iterable[EvolverTurnLightOn]:
        lights = state.by(Device, lambda d: d.dstate is DeviceState.OFF)
        for light in lights:
            yield cls(light=light)

    def then(self, state: WorldSnapshot) -> tuple[WorldSnapshot, int | float]:
        new_light = self._light.replace(dstate=DeviceState.ON)
        new_state = state.copy(add=[new_light], remove=[self._light])
        return new_state, 1


class EvolverTurnLightOff(WorldEvolver[WorldSnapshot, Action]):
    def __init__(self, light: Device) -> None:
        super().__init__(Action(device=cast(DeviceName, light.name), signal=ControlSignal.TURN_OFF))
        self._light: Device = light

    @classmethod
    def when(cls, state: WorldSnapshot) -> Iterable[EvolverTurnLightOff]:
        # Precondition: Light must currently be ON in the graph
        lights = state.by(Device, lambda d: d.dstate is DeviceState.ON)
        for light in lights:
            yield cls(light=light)

    def then(self, state: WorldSnapshot) -> tuple[WorldSnapshot, int | float]:
        new_light = self._light.replace(dstate=DeviceState.OFF)
        new_state = state.copy(add=[new_light], remove=[self._light])
        return new_state, 1


class EvolverLock(WorldEvolver[WorldSnapshot, Action]):
    def __init__(self, lock: Device) -> None:
        super().__init__(Action(device=cast(DeviceName, lock.name), signal=ControlSignal.LOCK))
        self._lock: Device = lock

    @classmethod
    def when(cls, state: WorldSnapshot) -> Iterable[EvolverLock]:
        # Precondition: Entity must be lockable and currently UNLOCKED
        locks = state.by(Device, lambda d: DeviceType.LOCKABLE in d.dtype and d.dstate is DeviceState.UNLOCKED)
        for lock in locks:
            yield cls(lock=lock)

    def then(self, state: WorldSnapshot) -> tuple[WorldSnapshot, int | float]:
        new_lock = self._lock.replace(dstate=DeviceState.LOCKED)
        new_state = state.copy(add=[new_lock], remove=[self._lock])
        return new_state, 1


class EvolverUnlock(WorldEvolver[WorldSnapshot, Action]):
    def __init__(self, lock: Device) -> None:
        super().__init__(Action(device=cast(DeviceName, lock.name), signal=ControlSignal.UNLOCK))
        self._lock: Device = lock

    @classmethod
    def when(cls, state: WorldSnapshot) -> Iterable[EvolverUnlock]:
        locks = state.by(Device, lambda d: DeviceType.LOCKABLE in d.dtype and d.dstate is DeviceState.LOCKED)
        for lock in locks:
            yield cls(lock=lock)

    def then(self, state: WorldSnapshot) -> tuple[WorldSnapshot, int | float]:
        new_lock = self._lock.replace(dstate=DeviceState.UNLOCKED)
        new_state = state.copy(add=[new_lock], remove=[self._lock])
        return new_state, 1


class EvolverStartAppliance(WorldEvolver[WorldSnapshot, Action]):
    def __init__(self, appliance: Device) -> None:
        super().__init__(Action(device=cast(DeviceName, appliance.name), signal=ControlSignal.START))
        self._appliance: Device = appliance

    @classmethod
    def when(cls, state: WorldSnapshot) -> Iterable[EvolverStartAppliance]:
        appliances = state.by(Device, lambda d: d.dstate is DeviceState.IDLE)
        for appliance in appliances:
            yield cls(appliance=appliance)

    def then(self, state: WorldSnapshot) -> tuple[WorldSnapshot, int | float]:
        new_appliance = self._appliance.replace(dstate=DeviceState.RUNNING)
        new_state = state.copy(add=[new_appliance], remove=[self._appliance])
        return new_state, 1


class EvolverStopAppliance(WorldEvolver[WorldSnapshot, Action]):
    def __init__(self, appliance: Device) -> None:
        super().__init__(Action(device=cast(DeviceName, appliance.name), signal=ControlSignal.STOP))
        self._appliance: Device = appliance

    @classmethod
    def when(cls, state: WorldSnapshot) -> Iterable[EvolverStopAppliance]:
        appliances = state.by(Device, lambda d: d.dstate is DeviceState.RUNNING)
        for appliance in appliances:
            yield cls(appliance=appliance)

    def then(self, state: WorldSnapshot) -> tuple[WorldSnapshot, int | float]:
        new_appliance = self._appliance.replace(dstate=DeviceState.IDLE)
        new_state = state.copy(add=[new_appliance], remove=[self._appliance])
        return new_state, 1

Successfully connected to Smart Home Simulation!


## Step 2: Defining the Goal

The `WorldEvolver` constructs specify how the environment changes through agent actions. Next, we define our target outcome. In `cognition`, goals are designed as evaluation predicates, i.e., functions that inspect a `WorldSnapshot` and return `True` when desired conditions are satisfied. 

The `create_away_goal` factory function below constructs and returns this predicate as `goal_test`. When passed a `WorldSnapshot`, `goal_test` queries the snapshot for the `main_door` lock entity and all `LIGHTABLE` devices. It evaluates two criteria: verifying that every lightable device is off and confirming that the main door is locked, returning `True` only when both conditions are fulfilled.   

In [2]:
from cognition import WorldSnapshot, Predicate


def away_goal_test(snapshot: WorldSnapshot) -> bool:
    # Query entities directly from the WorldSnapshot knowledge graph
    main_door_lock, *_ = snapshot.by(Device, lambda e: e.name == "main_door")
    lights = snapshot.by(Device, lambda e: DeviceType.LIGHTABLE in e.dtype)

    # Evaluate target state requirements
    all_lights_off = all(light.dstate is DeviceState.OFF for light in lights)
    is_door_locked = main_door_lock.dstate is DeviceState.LOCKED

    return all_lights_off and is_door_locked

## Step 3: Constructing and Solving the Reasoning Problem with `SearchPlanner`

With our causal `WorldEvolver` transitions established in Step 1 and our goal evaluation predicate designed in Step 2, we can now assemble the complete planning problem. 

In `cognition`, [automated planning](https://www.youtube.com/watch?v=epXjq1ekqao) is integrated directly into the agent's decision loop using standard `Operator` classes. The `PlanWithGoal` operator below triggers whenever a goal predicate is assigned to `state.goal` but no `plan` sequence has been derived yet. 

When `perform()` is invoked, `PlanWithGoal` passes three core elements to `SearchPlanner`:   
- Current State: The active `WorldSnapshot` knowledge graph.   
- Goal State: The goal evaluation predicate initialized with the current snapshot (`state.goal(snapshot)`).    
- Action Space: The suite of causal `WorldEvolver` constructs combined via `dynamic_opts_succession`. 

`SearchPlanner` simulates state transitions through the causal world model to discover an optimal sequence of actions, which are then stored in `state.plan` and dispatched directly to physical actuators. 

In [3]:
from typing import Any

from cognition import Operator, IOContainer, SearchPlanner, dynamic_opts_succession
from smart_home_assistant_v6 import HomeState


### HOW CAN I INIT THE PLANNER IN INIT

class PlanWithGoal(Operator[HomeState]):
    """Invokes SearchPlanner to automatically discover and execute action sequences."""

    def __init__(self, name: str, **kwargs: Any) -> None:
        self._succession = dynamic_opts_succession(
                        EvolverTurnLightOff,
                        EvolverTurnLightOn,
                        EvolverLock,
                        EvolverUnlock,
                        EvolverStartAppliance,
                        EvolverStopAppliance,
                    )

    def can_perform(self, state: HomeState, io: IOContainer) -> bool:
        # Trigger when a goal is assigned but no plan sequence has been derived yet
        return state.goal is not None and state.plan is None

    def perform(self, state: HomeState, io: IOContainer) -> None:
        print("--> [Planning] -> Searching world-graph transitions for goal path...")
        snapshot = state.world.snapshot
        # Construct planner using current WorldSnapshot, goal predicate, and dynamic operators
        planner = SearchPlanner(
            snapshot,
            state.goal(snapshot),
            self._succession
        )
        planner.run()
        print(f"--> [Planning] -> Derived plan ({planner.plan_found}) with {planner.plan_cost} action steps")

        # Store derived plan on state and dispatch actions to physical actuators
        state.plan = planner.plan
        for act in state.plan:
            print(f"--> [Executing] -> {act}")
            io.o.devices(act)

## Step 4: Connecting Goal-Driven Planning with `AWAY` Mode

To integrate goal-driven planning into our smart home assistant, we first extend the `Mode` enum to include an `AWAY` operating mode.   

When a resident requests away mode, the NLU pipeline and `SetMode` operator update `state.mode` to `Mode.AWAY`. This transition activates the `SetGoal` operator, which assigns our `create_away_goal` predicate (from Step 2) to `state.goal`. Once assigned, `PlanWithGoal` (from Step 3) detects the active goal and invokes `SearchPlanner` to derive and execute the plan.   

In [4]:
from cognition import AutoDocEnum, IOContainer, Operator
from smart_home_assistant_v6 import HomeState


class Mode(AutoDocEnum):
    """Operating modes for the home assistant."""

    DAY = "day mode of smart assistant"
    EVENING = "evening mode of smart assistant"
    MIDNIGHT = "midnight mode of smart assistant"
    AWAY = "away mode of smart assistant"


class SetGoal(Operator[HomeState]):
    """Assigns the away goal predicate when away mode is active."""

    def can_perform(self, state: HomeState, io: IOContainer) -> bool:
        # Trigger precondition: away mode active and no goal assigned yet
        return state.mode is Mode.AWAY and state.goal is None

    def perform(self, state: HomeState, io: IOContainer) -> None:
        state.goal = goal_test
        print("--> [Set Goal] -> Goal assigned: create_away_goal")

## Complete Planning Execution Lifecycle
1. Mode Request: A resident requests away mode, causing `SetMode` to update state.`mode = Mode.AWAY`. 
2. Goal Binding: SetGoal detects `Mode.AWAY` and sets `state.goal = create_away_goal`.   
3. Plan Generation: `PlanWithGoal` detects `state.goal`, initializes `SearchPlanner` with the causal `WorldEvolver` domain, and computes the action path.   
4. Physical Dispatch: `PlanWithGoal` iterates through the derived sequence and dispatches low-level actions to actuators via `io.o.devices(act)`.

## Step 5: Try Out the Full Agent
Now that our agent connects world model evolvers, goal predicates, and planning operators, let's test it in action:

1. Launch the Agent: Start the smart home simulation server and run `smart_home_assistant_v6.py` below. The assistant initializes in `Mode.DAY`.

2. Activate Appliances: Use the device controls or chat interface to turn on several lights and start both the washer and dryer.

3. Trigger Away Mode: In the chat interface, send the request: "Set mode to away".

### What Happens?
The assistant receives the mode change request, assigns the away goal predicate, and invokes `SearchPlanner`. The planner derives a sequence of actions that turns off every active light and locks the main entrance door and then finishes execution.

### Why Do the Washer and Dryer Keep Running?
`cognition` agent's reasoning operate strictly within the bounds of the goal predicate defined in Step 2 (`create_away_goal`).

In [9]:
!python smart_home_assistant_v6.py

Successfully connected to Smart Home Simulation!
Cogent running. Click 'Interrupt Kernel' in Jupyter or press Ctrl+C in terminal to stop.

--> [Update internal world model] -> setting bathroom to off
--> [Update internal world model] -> setting dishwasher to idle
--> [Interpret Intent triggered]
--> [Process intent decision process] -> recognized intent as ResidentIntent.SET_MODE
--> [Process intent decision process] -> recognized mode as Mode.AWAY
--> [Interpreted Intent] human expressed: StructuredIntent(intent=<ResidentIntent.SET_MODE: 'set_mode'>, object=<Mode.AWAY: 4>)
--> [Internal Action Triggered] Setting mode to Mode.AWAY
--> [Set goal] -> setting goal to None
--> [Planning] -> searching for how to achieve the goal
--> [Planning] -> found a plan True with 3 actions
--> [Executing] -> Action(device='living_room', signal=<ControlSignal.TURN_OFF: 'turn_off'>)
--> [Executing] -> Action(device='bedroom', signal=<ControlSignal.TURN_OFF: 'turn_off'>)
--> [Executing] -> Action(device=